# PIRLS16: Model Benchmarking & Feature Importance

This notebook trains and fine-tunes multiple tabular ML models on PIRLS16 data and summarizes performance (MAE/MSE) and feature importance. It reuses the existing preprocessing logic from the quick model notebook and adds a richer, reproducible benchmarking pipeline.

## Setup
Install required libraries (run once).

In [13]:
# If needed, install packages (safe to re-run)
%pip -q install pyreadstat scikit-learn lightgbm xgboost catboost tabpfn altair

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyreadstat

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyRegressor

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from tabpfn import TabPFNRegressor

import altair as alt
alt.renderers.enable('mimetype')

RendererRegistry.enable('mimetype')

In [15]:
# Fast mode to speed up experimentation
FAST_MODE = True
N_ITER = 4 if FAST_MODE else 12
CV_FOLDS = 2 if FAST_MODE else 3

# Optionally cap sample size in fast mode
if FAST_MODE:
    SAMPLE_N = 10000  # override earlier setting for faster runs

## Load Data

In [16]:
df_school, meta_school = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_SCHOOL16.sav')
df_student, meta_student = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STUDENT16.sav')
df_teacher, meta_teacher = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_TEACHER16.sav')
df_link, meta_link = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STD_TCH_LINK16.sav')

## Feature Engineering & Merge
Reuses the existing target creation and leakage-prone column removal from the quick model notebook.

In [17]:
# cols to generate y
y_gen_cols = ['ASRREA01', 'ASRREA02', 'ASRREA03', 'ASRREA04', 'ASRREA05']
# cols to drop because they might lead to leakage
cols_drop = y_gen_cols + ['ASRLIT01', 'ASRLIT02', 'ASRLIT03', 'ASRLIT04', 'ASRLIT05',
                        'ASRINF01', 'ASRINF02', 'ASRINF03', 'ASRINF04', 'ASRINF05',
                        'ASRIIE01', 'ASRIIE02', 'ASRIIE03', 'ASRIIE04', 'ASRIIE05',
                        'ASRRSI01', 'ASRRSI02', 'ASRRSI03', 'ASRRSI04', 'ASRRSI05',
                        'ASRIBM01', 'ASRIBM02', 'ASRIBM03', 'ASRIBM04', 'ASRIBM05']

In [18]:
def create_y(df, y_gen_cols):
    """Create y column by averaging the columns in y_gen_cols"""
    df = df.copy()
    df['y'] = df[y_gen_cols].mean(axis=1)
    return df

def remove_cols(df, cols_drop):
    """Remove columns from the dataframe if they exist"""
    df = df.copy()
    to_drop = [col for col in cols_drop if col in df.columns]
    return df.drop(columns=to_drop)

def merge_dfs(df_student, df_new, on):
    """Merge df_new into df_student on the given column"""
    df_student = df_student.copy()
    df_new = df_new.copy()
    original_rows = df_student.shape[0]
    cols_to_drop = [col for col in df_new.columns if col in df_student.columns and col != on]
    df_new = df_new.drop(columns=cols_to_drop)
    df_student = df_student.merge(df_new, on=on, how='left', suffixes=('', '_new'))
    cols_to_drop = [col for col in df_student.columns if col.endswith('_new')]
    df_student = df_student.drop(columns=cols_to_drop)
    assert df_student.shape[0] == original_rows, f"Rows changed from {original_rows} to {df_student.shape[0]}"
    return df_student

def create_X_and_y(df_student, df_teacher, df_school, df_link):
    df_student = create_y(df_student, y_gen_cols)
    df_student = remove_cols(df_student, cols_drop)
    df_school = remove_cols(df_school, cols_drop)
    df_teacher = remove_cols(df_teacher, cols_drop)
    df_link = remove_cols(df_link, cols_drop)

    df = merge_dfs(df_student, df_school, on='IDSCHOOL')
    df = merge_dfs(df, df_link, on='IDSTUD')
    df = merge_dfs(df, df_teacher, on='IDTEALIN')

    y = df['y']
    X = df.drop(columns=['y'])
    return X, y

In [19]:
X, y = create_X_and_y(df_student, df_teacher, df_school, df_link)
X.shape, y.shape

((4425, 385), (4425,))

## Train/Validation/Test Split
Optional: set `SAMPLE_N` to a smaller number for faster experimentation.

In [20]:
SEED = 9527
try:
    SAMPLE_N
except NameError:
    SAMPLE_N = None  # e.g., 15000 for a quick run

if SAMPLE_N is not None and len(X) > SAMPLE_N:
    idx = np.random.default_rng(SEED).choice(len(X), size=SAMPLE_N, replace=False)
    X = X.iloc[idx].reset_index(drop=True)
    y = y.iloc[idx].reset_index(drop=True)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)
X_train.shape, X_valid.shape, X_test.shape

((3097, 385), (664, 385), (664, 385))

## Preprocessing
- Drop high-missing columns, IDs, constants, and date column (same as quick model).
- Separate numeric/categorical columns and apply imputers + scaling/encoding.

In [21]:
# identify columns to drop based on missing rate
drop_cols = []
missing_rates = (X_train.isnull().sum() / X_train.shape[0]).sort_values(ascending=False)[:20]
miss_rate_bar = 0.8
missing_cols = missing_rates[missing_rates > miss_rate_bar].index.tolist()
drop_cols += missing_cols

# ID columns
id_cols = [col for col in X_train.columns if 'ID' in col]
drop_cols += id_cols

# date column
drop_cols += ['ITDATE']

def get_constant_cols(X):
    const_cols = X.nunique()[X.nunique() == 1].index.tolist()
    const_cols = [col for col in const_cols if X[col].isnull().sum() == 0]
    return const_cols

const_cols = get_constant_cols(X_train)
drop_cols += const_cols

class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
        self.is_fitted = False

    def fit(self, X, y=None):
        self.is_fitted = True
        return self

    def transform(self, X):
        assert self.is_fitted, 'ColumnDropper not fitted.'
        X_drop = X.drop(columns=self.drop_cols)
        self.remaining_cols = X_drop.columns
        return X_drop

pre_dropper = ColumnDropper(drop_cols)
X_train_d = pre_dropper.fit_transform(X_train)

numeric_features = X_train_d.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X_train_d.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
 )

def build_preprocess_pipeline():
    return Pipeline(steps=[('dropper', ColumnDropper(drop_cols)),
                           ('preprocessor', preprocessor)])

## Model Tuning & Evaluation
We use randomized search to tune hyperparameters and then evaluate on the test set.

In [22]:
def evaluate_model(estimator, param_distributions=None, n_iter=N_ITER):
    preproc = build_preprocess_pipeline()
    pipe = Pipeline(steps=[('pre', preproc), ('model', estimator)])

    if param_distributions:
        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_distributions,
            n_iter=n_iter,
            scoring='neg_mean_absolute_error',
            cv=CV_FOLDS,
            random_state=SEED,
            n_jobs=-1
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
    else:
        best_model = pipe.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    return best_model, mae, mse

In [ ]:
if FAST_MODE:
    rf_estimators = [200]
    et_estimators = [200]
    gb_estimators = [200]
    xgb_estimators = 300
    cat_iters = [300]
    max_depths = [4, 6]
    min_samples_split = [2, 5]
    min_samples_leaf = [1, 2]
    num_leaves = [31, 63]
    min_child_samples = [20, 50]
    learning_rates = [0.05, 0.1]
    c_values = [1, 10]
    eps_values = [0.1, 0.2]
    knn_neighbors = [5, 10]
else:
    rf_estimators = [200, 400, 800]
    et_estimators = [200, 400, 800]
    gb_estimators = [200, 400]
    xgb_estimators = 500
    cat_iters = [400, 800]
    max_depths = [3, 5, 8, 12, None]
    min_samples_split = [2, 5, 10, 20]
    min_samples_leaf = [1, 2, 5, 10]
    num_leaves = [31, 63, 127]
    min_child_samples = [10, 20, 50]
    learning_rates = [0.03, 0.05, 0.1]
    c_values = [1, 10, 50]
    eps_values = [0.1, 0.2, 0.5]
    knn_neighbors = [5, 10, 20]

lasso_max_iter = 20000 if FAST_MODE else 60000
elastic_max_iter = 20000 if FAST_MODE else 60000
lasso_tol = 1e-3 if FAST_MODE else 1e-4
elastic_tol = 1e-3 if FAST_MODE else 1e-4

models = {
    'DummyMean': (DummyRegressor(strategy='mean'), None),
    'LinearRegression': (LinearRegression(), None),
    'Ridge': (Ridge(), {'model__alpha': np.logspace(-3, 2, 20)}),
    'Lasso': (Lasso(max_iter=lasso_max_iter, tol=lasso_tol), {'model__alpha': np.logspace(-4, 1, 20)}),
    'ElasticNet': (ElasticNet(max_iter=elastic_max_iter, tol=elastic_tol), {'model__alpha': np.logspace(-4, 1, 20), 'model__l1_ratio': np.linspace(0.1, 0.9, 9)}),
    'DecisionTree': (DecisionTreeRegressor(random_state=SEED), {
        'model__max_depth': max_depths,
        'model__min_samples_split': min_samples_split,
        'model__min_samples_leaf': min_samples_leaf
    }),
    'RandomForest': (RandomForestRegressor(random_state=SEED, n_jobs=-1), {
        'model__n_estimators': rf_estimators,
        'model__max_depth': [None, 10, 20] if not FAST_MODE else [None, 10],
        'model__min_samples_split': [2, 5, 10] if not FAST_MODE else [2, 5],
        'model__min_samples_leaf': [1, 2, 4] if not FAST_MODE else [1, 2]
    }),
    'ExtraTrees': (ExtraTreesRegressor(random_state=SEED, n_jobs=-1), {
        'model__n_estimators': et_estimators,
        'model__max_depth': [None, 10, 20] if not FAST_MODE else [None, 10],
        'model__min_samples_split': [2, 5, 10] if not FAST_MODE else [2, 5],
        'model__min_samples_leaf': [1, 2, 4] if not FAST_MODE else [1, 2]
    }),
    'GradientBoosting': (GradientBoostingRegressor(random_state=SEED), {
        'model__n_estimators': gb_estimators,
        'model__learning_rate': learning_rates,
        'model__max_depth': [2, 3, 5] if not FAST_MODE else [2, 3]
    }),
    'SVR': (SVR(), {
        'model__C': c_values,
        'model__gamma': ['scale', 'auto'],
        'model__epsilon': eps_values
    }),
    'KNN': (KNeighborsRegressor(), {
        'model__n_neighbors': knn_neighbors,
        'model__weights': ['uniform', 'distance']
    }),
    'XGBoost': (XGBRegressor(
        random_state=SEED,
        n_estimators=xgb_estimators,
        tree_method='hist',
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        n_jobs=-1
    ), {
        'model__max_depth': [4, 6, 8] if not FAST_MODE else [4, 6],
        'model__subsample': [0.7, 0.9, 1.0] if not FAST_MODE else [0.8, 1.0],
        'model__colsample_bytree': [0.7, 0.9, 1.0] if not FAST_MODE else [0.8, 1.0],
        'model__min_child_weight': [1, 5, 10] if not FAST_MODE else [1, 5]
    }),
    'LightGBM': (LGBMRegressor(), None),
    'CatBoost': (CatBoostRegressor(
        loss_function='RMSE',
        random_seed=SEED,
        verbose=False
    ), {
        'model__depth': [6, 8, 10] if not FAST_MODE else [6, 8],
        'model__learning_rate': learning_rates,
        'model__iterations': cat_iters
    })
}

In [24]:
results = []
fitted_models = {}

for model_name, (estimator, params) in models.items():
    print(f'Training {model_name}...')
    best_model, mae, mse = evaluate_model(estimator, params, n_iter=12)
    fitted_models[model_name] = best_model
    results.append({'model': model_name, 'mae': mae, 'mse': mse})

results_df = pd.DataFrame(results).sort_values('mae').reset_index(drop=True)
results_df

Training LinearRegression...
Training Ridge...
Training Lasso...


/home/roc/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.464e+04, tolerance: 9.145e+03
  model = cd_fast.enet_coordinate_descent(
/home/roc/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.209e+05, tolerance: 8.731e+03
  model = cd_fast.enet_coordinate_descent(
/home/roc/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.910e+04,

Training ElasticNet...


/home/roc/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.338e+06, tolerance: 8.731e+03
  model = cd_fast.enet_coordinate_descent(
/home/roc/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.279e+06, tolerance: 8.731e+03
  model = cd_fast.enet_coordinate_descent(
/home/roc/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.032e+06,

Training DecisionTree...
Training RandomForest...
Training ExtraTrees...
Training GradientBoosting...
Training SVR...
Training KNN...
Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001345 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4496
[LightGBM] [Info] Number of data points in the train set: 3097, number of used features: 362
[LightGBM] [Info] Start training from score 548.731639
Training CatBoost...


,model,mae,mse
0,XGBoost,40.151288,2552.114223
1,CatBoost,40.558675,2610.344765
2,GradientBoosting,40.593363,2620.813586
3,LightGBM,41.326577,2637.570728
4,Ridge,41.436748,2840.752848
5,LinearRegression,41.713568,2894.971169
6,Lasso,41.845379,2861.371513
7,ElasticNet,42.190712,2928.325916
8,SVR,43.001869,3098.661701
9,RandomForest,43.604144,3024.354449


## TabPFN (Zero-Shot)
TabPFN is a foundation model for tabular data. We use it as a zero-shot regressor (no fine-tuning).

In [25]:
# Prepare data for TabPFN (dense arrays)
tab_preproc = build_preprocess_pipeline()
X_train_tab = tab_preproc.fit_transform(X_train)
X_test_tab = tab_preproc.transform(X_test)

# convert to dense if needed
if hasattr(X_train_tab, 'toarray'):
    X_train_tab = X_train_tab.toarray()
    X_test_tab = X_test_tab.toarray()

tabpfn = TabPFNRegressor(device='auto')
tabpfn.fit(X_train_tab, y_train)
tab_pred = tabpfn.predict(X_test_tab)
tab_mae = mean_absolute_error(y_test, tab_pred)
tab_mse = mean_squared_error(y_test, tab_pred)

results_df = pd.concat([
    results_df,
    pd.DataFrame([{'model': 'TabPFN', 'mae': tab_mae, 'mse': tab_mse}])
]).sort_values('mae').reset_index(drop=True)
results_df

tabpfn-v2-regressor.ckpt:   0%|          | 0.00/44.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

,model,mae,mse
0,XGBoost,40.151288,2552.114223
1,TabPFN,40.233365,2582.179805
2,CatBoost,40.558675,2610.344765
3,GradientBoosting,40.593363,2620.813586
4,LightGBM,41.326577,2637.570728
5,Ridge,41.436748,2840.752848
6,LinearRegression,41.713568,2894.971169
7,Lasso,41.845379,2861.371513
8,ElasticNet,42.190712,2928.325916
9,SVR,43.001869,3098.661701


## Feature Importance
We use model-native importance when available (`feature_importances_` or `coef_`). Otherwise we compute permutation importance on the test set.

In [27]:
def get_feature_names(preproc_pipeline):
    ct = preproc_pipeline.named_steps['preprocessor']
    try:
        feature_names = ct.get_feature_names_out()
        return np.array(feature_names)
    except Exception:
        feature_names = []
        num_feats = ct.transformers_[0][2]
        feature_names.extend(num_feats)
        if len(ct.transformers_) > 1:
            cat_ohe = ct.transformers_[1][1].named_steps['onehot']
            cat_feats = ct.transformers_[1][2]
            if hasattr(cat_ohe, 'categories_'):
                cat_names = cat_ohe.get_feature_names_out(cat_feats).tolist()
                feature_names.extend(cat_names)
        return np.array(feature_names)

def compute_importance(fitted_model, X_test=X_test, y_test=y_test, top_n=25):
    preproc = fitted_model.named_steps['pre']
    model = fitted_model.named_steps['model']
    X_test_trans = preproc.transform(X_test)
    if hasattr(X_test_trans, 'toarray'):
        X_test_dense = X_test_trans.toarray()
    else:
        X_test_dense = X_test_trans

    feature_names = get_feature_names(preproc)

    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_)
    else:
        pi = permutation_importance(model, X_test_dense, y_test, n_repeats=8, random_state=SEED, n_jobs=-1)
        importances = pi.importances_mean

    imp_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)

    return imp_df.head(top_n)

def plot_importance(imp_df, title):
    return alt.Chart(imp_df).mark_bar().encode(
        x=alt.X('importance:Q'),
        y=alt.Y('feature:O', sort='-x')
    ).properties(title=title, width=700, height=400)

importance_tables = {}
for name, fitted in fitted_models.items():
    imp_df = compute_importance(fitted)
    importance_tables[name] = imp_df
    display(plot_importance(imp_df, f'{name} - Top Features'))

# TabPFN importance via permutation (on transformed data)
tab_feature_names = get_feature_names(tab_preproc)
pi_tab = permutation_importance(tabpfn, X_test_tab, y_test, n_repeats=8, random_state=SEED, n_jobs=-1)
tab_imp_df = pd.DataFrame({'feature': tab_feature_names, 'importance': pi_tab.importances_mean})
tab_imp_df = tab_imp_df.sort_values('importance', ascending=False).head(25)
display(plot_importance(tab_imp_df, 'TabPFN - Top Features'))

importance_tables['TabPFN'] = tab_imp_df

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


...

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


/home/roc/.local/lib/python3.11/site-packages/tabpfn/inference.py:301: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  X_train = torch.as_tensor(X_train, dtype=torch.float32, device=device)  # noqa: PLW2901
/home/roc/.local/lib/python3.11/site-packages/tabpfn/inference.py:301: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. 

: 

## Final Results
The table below summarizes MAE and MSE for all fine-tuned models (including TabPFN).

In [ ]:
results_df

## Optional: Logistic Regression (Classification)
Logistic Regression is a classification model and does not directly optimize MAE/MSE for continuous targets. If you want a classification baseline, you can binarize `y` (e.g., above/below median) and evaluate classification metrics separately.